In [5]:
import requests
import pandas as pd

parcs = {
    4: "Disneyland Park Paris",
    28: "Disney Adventure World Paris",
    9: "Parc Astérix",
    291: "Futuroscope",
    301: "Walibi Rhône-Alpes",
    303: "Le Pal",
    336: "Nigloland",
    304: "Vulcania",
    327: "Le Parc du Petit Prince",
    320: "PanoraMagique",
}

url = "https://queue-times.com/parks/4/queue_times.json"
reponse = requests.get(url, timeout=10) # Va chercher les données à cette adresse, attends 10 secondes maximum, et range le résultat dans reponse.

print("Code de statut :", reponse.status_code)
data = reponse.json()
print("Clés du JSON :", data.keys())
print("Nombre de zones (lands) :", len(data["lands"]))
print("Attractions hors zone (rides) :", len(data["rides"]))


Code de statut : 200
Clés du JSON : dict_keys(['lands', 'rides'])
Nombre de zones (lands) : 5
Attractions hors zone (rides) : 0


In [6]:
from datetime import datetime, timezone

def collecter_parc(park_id, park_nom):
    """Récupère les attractions d'un parc et renvoie une liste de lignes."""
    url = f"https://queue-times.com/parks/{park_id}/queue_times.json"

    # Contrôle des erreurs : si l'API ne répond pas ou renvoie une erreur, on ne plante pas
    try:
        reponse = requests.get(url, timeout=10)
        reponse.raise_for_status()   # déclenche une erreur si le code n'est pas 200
    except requests.exceptions.RequestException as erreur:
        print(f"Erreur pour {park_nom} : {erreur}")
        return []

    data = reponse.json()
    heure_collecte = datetime.now(timezone.utc)   # moment où NOTRE script a collecté
    lignes = []

    # Cas 1 : attractions rangées dans des zones (ex. Disneyland)
    for zone in data["lands"]:
        for attraction in zone["rides"]:
            lignes.append({
                "park_id": park_id,
                "park_nom": park_nom,
                "zone": zone["name"],
                "ride_id": attraction["id"],
                "ride_nom": attraction["name"],
                "is_open": attraction["is_open"],
                "wait_time": attraction["wait_time"],
                "last_updated": attraction["last_updated"],
                "heure_collecte": heure_collecte,
            })

    # Cas 2 : attractions hors zone (ex. Futuroscope)
    for attraction in data["rides"]:
        lignes.append({
            "park_id": park_id,
            "park_nom": park_nom,
            "zone": None,
            "ride_id": attraction["id"],
            "ride_nom": attraction["name"],
            "is_open": attraction["is_open"],
            "wait_time": attraction["wait_time"],
            "last_updated": attraction["last_updated"],
            "heure_collecte": heure_collecte,
        })

    print(f"{park_nom} : {len(lignes)} attractions récupérées")
    return lignes


# On boucle sur les 4 parcs et on rassemble tout
toutes_les_lignes = []
for park_id, park_nom in parcs.items():
    toutes_les_lignes.extend(collecter_parc(park_id, park_nom))

df = pd.DataFrame(toutes_les_lignes)
df.head(10)

Disneyland Park Paris : 42 attractions récupérées
Disney Adventure World Paris : 22 attractions récupérées
Parc Astérix : 50 attractions récupérées
Futuroscope : 56 attractions récupérées
Walibi Rhône-Alpes : 27 attractions récupérées
Le Pal : 29 attractions récupérées
Nigloland : 39 attractions récupérées
Vulcania : 0 attractions récupérées
Le Parc du Petit Prince : 28 attractions récupérées
PanoraMagique : 1 attractions récupérées


,park_id,park_nom,zone,ride_id,ride_nom,is_open,wait_time,last_updated,heure_collecte
0,4,Disneyland Park Paris,Adventureland,2702,Adventure Isle,True,0,2026-10-12T10:25:04.000Z,2026-10-12 10:27:37.737670+00:00
1,4,Disneyland Park Paris,Adventureland,7306,Indiana Jones and the Temple of Peril Single R...,True,30,2026-10-12T10:25:04.000Z,2026-10-12 10:27:37.737670+00:00
2,4,Disneyland Park Paris,Adventureland,2,Indiana Jones™ and the Temple of Peril,True,45,2026-10-12T10:25:04.000Z,2026-10-12 10:27:37.737670+00:00
3,4,Disneyland Park Paris,Adventureland,1,La Cabane des Robinson,True,5,2026-10-12T10:25:04.000Z,2026-10-12 10:27:37.737670+00:00
4,4,Disneyland Park Paris,Adventureland,2703,Le Passage Enchanté d'Aladdin,True,0,2026-10-12T10:25:04.000Z,2026-10-12 10:27:37.737670+00:00
5,4,Disneyland Park Paris,Adventureland,2705,Pirate Galleon,False,0,2026-10-12T10:25:04.000Z,2026-10-12 10:27:37.737670+00:00
6,4,Disneyland Park Paris,Adventureland,3,Pirates of the Caribbean,True,40,2026-10-12T10:25:04.000Z,2026-10-12 10:27:37.737670+00:00
7,4,Disneyland Park Paris,Adventureland,2704,Pirates' Beach,False,0,2026-10-12T10:25:04.000Z,2026-10-12 10:27:37.737670+00:00
8,4,Disneyland Park Paris,Discoveryland,4,"Autopia, presented by Avis",True,35,2026-10-12T10:25:04.000Z,2026-10-12 10:27:37.737670+00:00
9,4,Disneyland Park Paris,Discoveryland,5,Buzz Lightyear Laser Blast,False,0,2026-10-12T10:25:04.000Z,2026-10-12 10:27:37.737670+00:00


In [7]:
# Traduction des noms de colonnes en français (sans accents ni espaces, pour BigQuery)
df = df.rename(columns={
    "park_id": "id_parc",
    "park_nom": "nom_parc",
    "zone": "zone",
    "ride_id": "id_attraction",
    "ride_nom": "nom_attraction",
    "is_open": "est_ouverte",
    "wait_time": "temps_attente_min",
    "last_updated": "date_maj_api",
    "heure_collecte": "date_collecte",
})
df.head()

,id_parc,nom_parc,zone,id_attraction,nom_attraction,est_ouverte,temps_attente_min,date_maj_api,date_collecte
0,4,Disneyland Park Paris,Adventureland,2702,Adventure Isle,True,0,2026-10-12T10:25:04.000Z,2026-10-12 10:27:37.737670+00:00
1,4,Disneyland Park Paris,Adventureland,7306,Indiana Jones and the Temple of Peril Single R...,True,30,2026-10-12T10:25:04.000Z,2026-10-12 10:27:37.737670+00:00
2,4,Disneyland Park Paris,Adventureland,2,Indiana Jones™ and the Temple of Peril,True,45,2026-10-12T10:25:04.000Z,2026-10-12 10:27:37.737670+00:00
3,4,Disneyland Park Paris,Adventureland,1,La Cabane des Robinson,True,5,2026-10-12T10:25:04.000Z,2026-10-12 10:27:37.737670+00:00
4,4,Disneyland Park Paris,Adventureland,2703,Le Passage Enchanté d'Aladdin,True,0,2026-10-12T10:25:04.000Z,2026-10-12 10:27:37.737670+00:00


In [8]:
# 1. Structure et types des colonnes
print("=== Types des colonnes ===")
df.info()

# 2. Valeurs manquantes par colonne
print("\n=== Valeurs manquantes ===")
print(df.isna().sum())

# 3. Doublons
print("\n=== Doublons (id_parc + id_attraction + date_maj_api) ===")
print(df.duplicated(subset=["id_parc", "id_attraction", "date_maj_api"]).sum())

# 4. Attractions ouvertes / fermées par parc
print("\n=== Ouvertes / fermées par parc ===")
print(df.groupby("nom_parc")["est_ouverte"].value_counts().unstack())

# 5. Temps d'attente quand l'attraction est fermée
print("\n=== temps_attente_min des attractions fermées ===")
print(df.loc[~df["est_ouverte"], "temps_attente_min"].value_counts())

# 6. Statistiques des temps d'attente (attractions ouvertes uniquement)
print("\n=== temps_attente_min des attractions ouvertes, par parc ===")
print(df[df["est_ouverte"]].groupby("nom_parc")["temps_attente_min"].describe())

# 7. Lignes "Single Rider"
print("\n=== Lignes Single Rider ===")
print(df[df["nom_attraction"].str.contains("Single Rider", case=False)][["nom_parc", "nom_attraction", "temps_attente_min"]])

=== Types des colonnes ===
<class 'pandas.DataFrame'>
RangeIndex: 294 entries, 0 to 293
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   id_parc            294 non-null    int64              
 1   nom_parc           294 non-null    str                
 2   zone               68 non-null     str                
 3   id_attraction      294 non-null    int64              
 4   nom_attraction     294 non-null    str                
 5   est_ouverte        294 non-null    bool               
 6   temps_attente_min  294 non-null    int64              
 7   date_maj_api       294 non-null    str                
 8   date_collecte      294 non-null    datetime64[us, UTC]
dtypes: bool(1), datetime64[us, UTC](1), int64(3), str(4)
memory usage: 18.8 KB

=== Valeurs manquantes ===
id_parc                0
nom_parc               0
zone                 226
id_attraction          0
nom_attracti

In [9]:
df_clean = df.copy()

# 1. Conversion de date_maj_api (texte) en vraie date UTC
df_clean["date_maj_api"] = pd.to_datetime(df_clean["date_maj_api"], utc=True)

# 2. Zones manquantes : absence à la source, pas une erreur
df_clean["zone"] = df_clean["zone"].fillna("Non renseignée")

# 3. Repérage des files Single Rider (gardées mais signalées)
df_clean["est_single_rider"] = df_clean["nom_attraction"].str.contains("Single Rider", case=False)

# 4. Suppression des doublons éventuels (clé : parc + attraction + date de l'API)
nb_avant = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["id_parc", "id_attraction", "date_maj_api"])
print("Doublons supprimés :", nb_avant - len(df_clean))

# Vérifications
df_clean.info()
print(df_clean["est_single_rider"].value_counts())

Doublons supprimés : 0
<class 'pandas.DataFrame'>
RangeIndex: 294 entries, 0 to 293
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   id_parc            294 non-null    int64              
 1   nom_parc           294 non-null    str                
 2   zone               294 non-null    str                
 3   id_attraction      294 non-null    int64              
 4   nom_attraction     294 non-null    str                
 5   est_ouverte        294 non-null    bool               
 6   temps_attente_min  294 non-null    int64              
 7   date_maj_api       294 non-null    datetime64[us, UTC]
 8   date_collecte      294 non-null    datetime64[us, UTC]
 9   est_single_rider   294 non-null    bool               
dtypes: bool(2), datetime64[us, UTC](2), int64(3), str(3)
memory usage: 19.1 KB
est_single_rider
False    285
True       9
Name: count, dtype: int64
